### **Data Reading**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@tjdatabricksete.dfs.core.windows.net/orders")

In [0]:
df = df.withColumn("year",year(df.order_date))
display(df)

In [0]:
df1 = df.withColumn("flag", dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))\
        .drop("_rescued_data")
df1.display()

### **Classes - OOP**

In [0]:
class windows:
    def dense_rank(self,df):
        df_dense_rank = df.withColumn("dense_rank",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_dense_rank
    
    def rank(self,df):
        df_rank = df.withColumn("rank",rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_rank
    
    def row_number(self,df):
        df_row_rank = df.withColumn("row_rank",row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_row_rank
  

In [0]:
df_new = df

In [0]:
obj = windows()
df_new = obj.dense_rank(df_new)
#df_new = obj.rank(df_new)
#df_new = obj.row_number(df_new)
df_new.display()
#df1 = df_new.filter(df_new.flag <= 3

### **Data Writing**

In [0]:
df1.write.format("delta").mode("overwrite").save("abfss://silver@tjdatabricksete.dfs.core.windows.net/orders")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksete_cat.silver.orders_silver
USING DELTA
LOCATION 'abfss://silver@tjdatabricksete.dfs.core.windows.net/orders'